# [7.2] Feature Verbalizers - Solutions

This notebook runs the reference implementations, local exercise tests, notebook contract, and report-backed signature result.

<details>
<summary>Help - what to compare with your exercise notebook</summary>

The reference solution is deliberately simple: whole-token keyword matching, train-only term learning, explicit counterexamples, signed intervention deltas, and a brevity check. The point is the validation loop, not a fancy verbalizer model.

</details>

<details>
<summary>Expected output</summary>

All local tests should pass, and the signature report should show the pinned `gelu-1l` preflight passing with held-out accuracy `1.0` and intervention delta above `0.5`.

</details>


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

chapter = "chapter7_activation_to_language"
section = "part2_feature_verbalizers"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_feature_verbalizers.solutions as solutions
import part2_feature_verbalizers.tests as tests


## Local Tests

The tests below cover both the happy path and the common verbalizer failure modes: substring matching, bad shapes, held-out leakage, ungrounded revisions, and signless intervention deltas.

<details>
<summary>Expected output</summary>

Each test prints `All tests ... passed!`, including `test_notebook_contract`.

</details>

<details>
<summary>Help - why so many small tests?</summary>

They are not mock evidence. They protect the individual moves in the learner loop so the final report means what it says.

</details>


In [ ]:
tests.test_gather_verbalizer_examples_covers_top_bottom_random_contrastive(
    solutions.gather_verbalizer_examples,
)
tests.test_gather_verbalizer_examples_rejects_bad_shapes_and_k(
    solutions.gather_verbalizer_examples,
)
tests.test_keyword_predictions_and_explanation_report_use_baseline_and_contrastives(
    solutions.keyword_explanation_predictions,
    solutions.explanation_prediction_report,
)
tests.test_keyword_predictions_reject_empty_explanation_terms(
    solutions.keyword_explanation_predictions,
)
tests.test_keyword_predictions_do_not_match_substrings(
    solutions.keyword_explanation_predictions,
)
tests.test_explanation_prediction_report_rejects_shape_mismatch(
    solutions.explanation_prediction_report,
)
tests.test_learned_verbalizer_terms_do_not_use_heldout_only_words(
    solutions.learn_verbalizer_terms,
)
tests.test_learned_verbalizer_terms_reject_bad_inputs(solutions.learn_verbalizer_terms)
tests.test_counterexamples_and_revision_are_grounded_in_failures(
    solutions.find_counterexamples,
    solutions.revise_explanation,
)
tests.test_counterexamples_reject_bad_inputs(solutions.find_counterexamples)
tests.test_intervention_prediction_checks_signed_direction(
    solutions.intervention_prediction_report,
)
tests.test_intervention_prediction_rejects_invalid_direction(
    solutions.intervention_prediction_report,
)
tests.test_explanation_brevity_compares_against_examples_only_baseline(
    solutions.explanation_brevity_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)


## Notebook Contract

<details>
<summary>Expected output</summary>

The contract should include top, bottom, random, and contrastive examples; prediction metrics; a grounded counterexample revision; an intervention report; and a brevity report.

</details>

<details>
<summary>Help - how to read the contract</summary>

The contract is the CPU version of the verbalizer loop. The CUDA report uses the same ideas on a real residual direction.

</details>


In [ ]:
contract = solutions.run_smoke_test(cpu=True)
contract


## Signature Result

The committed report is the real-model result for this section.

<details>
<summary>Expected output</summary>

`preflight_passed == True`, `prediction_accuracy == 1.0`, `contrastive_accuracy == 1.0`, `target_beats_random_intervention == True`, and `peak_vram_gb < 1.0`.

</details>

<details>
<summary>Help - interpreting the result</summary>

The combination matters: separation, held-out prediction, zero counterexamples on the small split, and target direction beating an orthogonal random control. Any one metric alone would be too weak.

</details>


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert gpu["preflight_passed"], "7.2 CUDA preflight should pass."
assert gpu["prediction_accuracy"] == 1.0
assert gpu["contrastive_accuracy"] == 1.0
assert gpu["target_beats_random_intervention"]
assert gpu["peak_vram_gb"] < 1.0
summary = {
    "explanation": gpu["explanation"],
    "top_examples": gpu["top_examples"],
    "bottom_examples": gpu["bottom_examples"],
    "heldout_contrastive_examples": gpu.get("heldout_contrastive_examples", []),
    "prediction_accuracy": gpu["prediction_accuracy"],
    "baseline_accuracy": gpu["baseline_accuracy"],
    "contrastive_accuracy": gpu["contrastive_accuracy"],
    "score_separation": gpu["score_separation"],
    "intervention_delta": gpu["intervention_delta"],
    "random_direction_intervention_delta": gpu["random_direction_intervention_delta"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}
summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].bar(["negative", "positive"], [gpu["negative_mean_score"], gpu["positive_mean_score"]], color=["#dc2626", "#16a34a"])
axes[0].axhline(0, color="#475569", linewidth=1)
axes[0].set_title("Projection scores")
axes[0].set_ylabel("mean score")
axes[1].bar(["target", "random"], [gpu["intervention_delta"], gpu["random_direction_intervention_delta"]], color=["#0891b2", "#94a3b8"])
axes[1].axhline(0.5, color="#64748b", linestyle="--", linewidth=1)
axes[1].set_title("Intervention deltas")
fig.tight_layout()
plt.show()


## Limitations

This solution validates a GT-1 residual-direction preflight, not a general feature-verbalizer benchmark. It does not claim SAE-feature explanations, API-LLM verbalization, broad OOD generalization, or causal control over generated completions.

## Further Research

Run multiple random-direction controls, subtract the target direction, test lexical adversaries that contain the learned terms, or apply the same loop to a Chapter 6 SAE feature.


In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu_report = report["metrics"]["gpu_test"]
    assert gpu_report["peak_vram_gb"] <= max_vram_gb
    return gpu_report


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


run_full_experiment(max_vram_gb=24.0)
